# Exercise: Categorical Variables

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

import sys
import os
sys.path.append(os.path.abspath('../..')) # add parent directory to search path for modules

from src.utilities import read_filter_data

# Read data, filter rows with missing target, separate target from predictors
X, X_test, y = read_filter_data()

# To keep things simple we'll drop columns with missing values
cols_with_missing = [
    col for col in X.columns
    if X[col].isnull().any()
]
X.drop(cols_with_missing, axis=1, inplace=True)
X_test.drop(cols_with_missing, axis=1, inplace=True)

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [ ]:
print(X_train.head())

drop columns with categorical data

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=0)
drop_X_train = X_train.select_dtypes(exclude=['object']) # exclude categorical variables (object)
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

In [ ]:
from src.utilities import score_model

print("MAE from Approach 1 (Drop categorical variables):")
print(score_model(model, drop_X_train, drop_X_valid, y_train, y_valid))

before jumping into ordinal encoding, we'll investigate the dataset. specifically we'll look at the 'Condition2' column. the code cell below prints the unique entries in both the training and validation sets.

In [ ]:
# unique() method returns an array of unique values in order of appearance, not frequency
print("Unique values in 'Condition2' column in training data:", X_train['Condition2'].unique())
print("\nUnique values in 'Condition2' column in validation data:", X_valid['Condition2'].unique())

if you write code now to:
- fit an ordinal encoder to the training data
- then use it to transform both the training and validation data

you'll get an error because the validation data contains unknown unique values. in this case, validation data Condition2's column contains the values 'RRAn' and 'RRNn', but these don't appear in the training data thus, if we try to use an ordinal encoder with scikit-learn, the code will throw an error

This is a common problem to encounter with real-world data, and there are many approache to fixing this issue. For instance, you can write a custom ordinal encoder to deal with new categories. The simplest approach, however, is to drop the problematic categorical columns.

In [ ]:
# Categorical columns in the training data
object_cols = [
    col for col in X_train.columns
    if X_train[col].dtype == "string"
]

# Columns that can be safely label encoded
good_label_cols = [ # list of columns that are safe to ordinal encode
    col for col in object_cols # loop through object columns
    if set(X_valid[col]).issubset(set(X_train[col])) # set(X_train[col]) gets unique values in X_train[col]
    # issubset(set(X_valid[col])) checks if all unique values in X_valid[col] are in X_train[col]
]

# Problematic columns that will be dropped from the dataset
bad_label_cols = list(set(object_cols)-set(good_label_cols)) # set subtraction
# all unique values in object_cols minus all unique values in good_label_cols = columns that are unsafe

print('Categorical columns that will be ordinal encoded:', good_label_cols)
print('\nCategorical columns that will be dropped from the dataset:', bad_label_cols)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Drop categorical columns that will not be encoded (unsafe)
label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_valid = X_valid.drop(bad_label_cols, axis=1)

# Apply ordinal encoding to each column
ordinal_encoder = OrdinalEncoder()
label_X_train[good_label_cols] = ordinal_encoder.fit_transform(X_train[good_label_cols]) # fit and transform
# encoder learns category mapping (fit) then replaces each category with respective integer (transform)
label_X_valid[good_label_cols] = ordinal_encoder.transform(X_valid[good_label_cols]) # transform valid data
# only transform valid data to avoid data leakage

In [ ]:
print("MAE from Approach 2 (Ordinal Encoding):")
print(score_model(model, label_X_train, label_X_valid, y_train, y_valid))

so far we've tried out two different approaches to dealing with categorical variables. One-hot encoding is next but before that there's one additional topic to cover:

In [ ]:
# Get number of unique entries in each column with categorical data
object_nunique = list( # list of sum of unique entries in each column
    map( # map(function, iterable), apply function to each element of the iterable
        lambda col: X_train[col].nunique(), object_cols
        # iterate through object_cols, and for each column it calculates X_train[col].nunique()
        # nunique() returns number of unique entries in the column
        # lambda is commonly used to create anoymous inline functions, 'lambda parameters: expression'
    )
)
d = dict(zip(object_cols, object_nunique)) # dictionary of object columns and respective sum of unique entries
# zip() combines elements of two iterables into tuples, pairing corresponding elements by index
# dict() then converts the resulting tuples (object_cols, object_nunique) into dictionary ('key': value)

# Print number of unique entries by column, in ascending order
sorted(d.items(), key=lambda x: x[1]) # sort dictionary 'd' by sum of unique entries (ascending)
# sorted() returns/prints sorted list, sorted(iterable, key=function, reverse=boolean), key is function to customize how to sort, default is 'None'
# items() turns dictionary into list of tuples [('key', value), ('key', value), ...]
# lambda x: x[1] for each tuple, use second element x[1] for sorting, in this case the sum of unique entries

the output above shows, for each column with categorical data, the number of unique values in the column. For instance, the 'Street' column in the training data has 2 unique values 'Grvl' and 'Pave'.

we refer to the number of unique entries of a categorical variable as the cardinality of that categorical variable. for instance the 'Street' variable has cardinality 2.

with this data, we can determine, for instance, there are 3 categorical variables in the training data that have cardinality > 10 ('Exterior1st', Exterior2nd', 'Neighborhood'). we also know 25 one-hot encoded columns are needed for 'Neighborhood' as it has 25 unique entries, i.e. we know how many columns are needed to one-hot encode each categorical variable (columns) in the training data.

for large datasets with many rows, one-hot encoding can greatly expand the size of the dataset. for this reason, we typically will only one-hot enocde columns with relatively low cardinality. then, high cardinality columns can be either dropped or ordinally encoded.

as an example, consider a dataset with 10,000 rows, and containing one categorical columns with 100 unique entries:
- if this column is replaced with the corresponding one-hot encoded columns: 10,000 (rows) * 100 (new columns) minus 10,000 (remove original column entries) = 999,000 new entries would be added to the dataset!
- if this column is replaced with ordinal encoding: 0 entries would be added as every unique value would be replaced by a corresponding integer

In [ ]:
# Columns that will be one-hot encoded
low_cardinality_cols = [ # list of columns < 10 nunique entries
    col for col in object_cols # loop through object columns
    if X_train[col].nunique() < 10 # if column has < 10 unique entries
]

# Columns that will be dropped from the dataset
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols)) # set subtraction
# all object columns minus all low cardinality columns = high cardinality columns

In [ ]:
print('Categorical columns that will be one-hot encoded:', low_cardinality_cols)
print('\nCategorical columns that will be dropped from the dataset:', high_cardinality_cols)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Apply one-hot encoder to each column with categorical data
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_X_train = pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols])) # fit and transform low cardinality columns in training data
OH_X_valid = pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols])) # transform validation data

# One-hot encoding removed index; put it back
OH_X_train.index = X_train.index
OH_X_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_X_train], axis=1) # combine numerical and one-hot encoded columns
OH_X_valid = pd.concat([num_X_valid, OH_X_valid], axis=1)

# Ensure all columns have type string
OH_X_train.columns = OH_X_train.columns.astype(str) # convert columns names to strings
OH_X_valid.columns = OH_X_valid.columns.astype(str)

In [ ]:
print("MAE from Approach 3 (One-Hot Encoding):")
print(score_model(model, OH_X_train, OH_X_valid, y_train, y_valid))


since imputation with the median value seems to yield more accurate results (lower MAE), we will be using it as that as the preferred method to use on X_test

In [ ]:
final_encoder = OH_encoder
final_X_train = OH_X_train

model.fit(final_X_train, y_train)

# Apply encoder to test data
final_OH_cols = pd.DataFrame(final_encoder.transform(X_test[low_cardinality_cols]))

# Ensure index remains same after encoding
final_OH_cols.index = X_test.index

# Remove categorical columns (will replace with one-hot encoding)
final_X_drop = X_test.drop(object_cols, axis=1)

# Combine numerical and one-hot encoded columns
final_X_test = pd.concat([final_X_drop, final_OH_cols], axis=1)

# Ensure test columns match training data, no need to convert to str as final_X_train columns already converted
final_X_test.columns = final_X_train.columns

# Predict on test data
preds_test = model.predict(final_X_test)

create a csv file with id and sales predictions below

In [ ]:
# create output dataframe with Id and SalePrice (predictions)
output = pd.DataFrame(
    {
        'Id': X_test.index, # create Id column, X_test.index are the house indices in X_test
        'PredictedSalePrice': preds_test # create PredictedSalePrice column, preds_test are the predicted prices
    }
)

# uncomment the line below to create a csv file
# output.to_csv('submission.csv', index=False) # create csv from 'output' dataframe, index=False removes index column